# Stage 4: Validation Strategy & Split Generator Notebook
## Amazon ML Challenge — Business Entity Resolution

This notebook partitions the Source 1 entities into a leak-free 80/20 train/validation split stratified by match cardinality (singletons, 1-to-1, 1-to-many) and country. It enforces that $Train \cap Val = \emptyset$ at the entity level.

In [ ]:
import sys
import json
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.stage_runner import StageController
from src.utils.storage import StorageManager
from src.evaluation.split import run_validation_split

### Step 1: Generate & Persist Validation Split

In [ ]:
storage = StorageManager("../configs/config.yaml")
storage.initialize_directories()

# Run Stage 4 via StageController
controller = StageController("../configs/config.yaml")
success = controller.run_stage("validation", force=True)

### Step 2: Inspect Validation Partition Statistics

In [ ]:
split_file = storage.splits_dir / "validation_split.json"
if split_file.exists():
    with open(split_file, "r", encoding="utf-8") as f:
        split_data = json.load(f)
    
    print("=" * 80)
    print("               STAGE 4: VALIDATION SPLIT SUMMARY               ")
    print("=" * 80)
    print(f"Split Identifier  : {split_data['split_id']}")
    print(f"Random Seed       : {split_data['random_seed']}")
    print(f"Validation Ratio  : {split_data['val_ratio'] * 100}%")
    print(f"Total S1 Entities : {split_data['total_source1_entities']:,}")
    print(f"Train Entities    : {len(split_data['train_entity_ids']):,}")
    print(f"Val Entities      : {len(split_data['val_entity_ids']):,}")
    print("-" * 80)
    
    t = split_data.get('train_statistics', {})
    v = split_data.get('validation_statistics', {})
    print(f"Train Singletons  : {t.get('singletons_count', 0):,} ({t.get('singletons_pct', 0)}%)")
    print(f"Val Singletons    : {v.get('singletons_count', 0):,} ({v.get('singletons_pct', 0)}%)")
    print(f"Train 1-to-1      : {t.get('one_to_one_count', 0):,} ({t.get('one_to_one_pct', 0)}%)")
    print(f"Val 1-to-1        : {v.get('one_to_one_count', 0):,} ({v.get('one_to_one_pct', 0)}%)")
    print(f"Train 1-to-Many   : {t.get('one_to_many_count', 0):,} ({t.get('one_to_many_pct', 0)}%)")
    print(f"Val 1-to-Many     : {v.get('one_to_many_count', 0):,} ({v.get('one_to_many_pct', 0)}%)")
    print("=" * 80)
else:
    print("Validation split file not found.")